# GraphRAG: turn documents into a connected map — built from scratch

Flat vector RAG retrieves the top-k SIMILAR chunks and structurally fails on two question types:
**multi-hop** (connect facts across documents — no chunk holds the chain) and **global/thematic**
(the answer is an aggregate over the whole corpus). GraphRAG (Edge et al. 2024) fixes both by turning
the pile of documents into a **knowledge graph**, then answering by **traversal** (local) or
**community summarization** (global). This notebook builds and **measures** each on a REAL graph.

> **Honesty (read first):** a real GraphRAG pipeline uses an LLM to EXTRACT entities/relations and to
> WRITE community summaries. This env is encoder-only, so we do NOT fake an LLM. What runs **real and
> measured**: the knowledge graph is a real `networkx` graph, the multi-hop traversal is a real
> shortest-path, the community detection is real modularity optimization, the modularity score Q is
> real, and the flat-RAG baseline is the real `all-MiniLM-L6-v2` dense retriever (ch5). **Illustrative
> (labelled)**: the entity/relation extraction and the community summaries (fixed hand-authored
> exemplars standing in for the LLM). Every graph statistic, path, community count, and modularity
> number below is measured and asserted.

## Step 1 — set up: import the canonical functions

Everything uses `graph_rag.py` (which reuses ch5's `DenseRetriever` + ch1's corpus). We print
numpy/networkx/torch versions for reproducibility; the encoder is CPU-pinned (deterministic).

In [1]:
import numpy as np
import networkx as nx

from graph_rag import (
    TOP_K,
    DenseRetriever,
    build_graph,
    build_multihop_probe,
    detect_communities,
    flat_rag_can_answer_multihop,
    full_corpus,
    global_search,
    graph_without_edge,
    modularity_score,
    path_with_relations,
    shortest_path,
)

print('numpy:', np.__version__)
print('networkx:', nx.__version__)
try:
    import torch
    avail = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
    print('torch:', torch.__version__, '| accelerator available:', avail, '| encoder runs on: cpu')
except ImportError:
    print('torch: not installed (graph algorithms are pure networkx — unaffected)')

numpy: 2.4.6
networkx: 3.6.1


torch: 2.12.0 | accelerator available: mps | encoder runs on: cpu


## Step 2 — build the knowledge graph (real networkx)

In production an LLM extracts (subject, relation, object) triples from each chunk. Here they are a
FIXED, labelled hand-authored set — but the graph it builds, and every algorithm below, are REAL.

In [2]:
graph = build_graph()
print(f'knowledge graph: {graph.number_of_nodes()} entities, {graph.number_of_edges()} relations')
print('entities:', sorted(graph.nodes()))
print('relations:')
for a, b in graph.edges():
    print(f'  {a} -[{graph.edges[a, b]["relation"]}]- {b}')
assert graph.number_of_nodes() == 10 and graph.number_of_edges() == 10, 'graph must have 10 entities, 10 relations'

knowledge graph: 10 entities, 10 relations
entities: ['Ariane 6', 'Dr. Amara Okoye', 'Error E-4011', 'Helios-7', 'Kourou spaceport', 'Nairobi office', 'ground team', 'hyperspectral imager', 'sun-synchronous orbit', 'telemetry stream']
relations:
  Helios-7 -[launched_from]- Kourou spaceport
  Helios-7 -[launched_aboard]- Ariane 6
  Helios-7 -[led_by]- Dr. Amara Okoye
  Helios-7 -[carries]- hyperspectral imager
  Helios-7 -[flies_in]- sun-synchronous orbit
  Helios-7 -[belongs_to]- telemetry stream
  Dr. Amara Okoye -[based_in]- Nairobi office
  Nairobi office -[operated_by]- ground team
  Error E-4011 -[appeared_in]- telemetry stream
  Error E-4011 -[investigates]- ground team


![The Helios-7 knowledge graph: 10 entities, 10 relations, coloured by detected community (Q=0.320).
Graph/traversal/community detection are real (networkx); the extraction is an illustrative LLM
stand-in. Generated by `make_figures_09.py`.](../../images/rag09_knowledge_graph.png)

## Step 3 — multi-hop: flat vector RAG can't CONNECT; graph traversal can

Ask *"Where is the team based that leads the satellite carrying the hyperspectral imager?"*. Flat RAG
retrieves the endpoints but NO single chunk links the imager to the office — so it can't connect them.
Graph traversal walks the real shortest path across three facts in three different chunks.

In [3]:
corpus = full_corpus()
retriever = DenseRetriever(corpus)
probe = build_multihop_probe()
print(f'query: {probe.query}')
print(f'dense lens: {retriever.backend}')
flat_hits = retriever.search(probe.query, k=TOP_K).indices
print(f'  flat-RAG top-{TOP_K} chunks: {list(flat_hits)}')
for idx in flat_hits:
    print(f'    chunk[{idx}]: {corpus[idx]}')
flat_ok = flat_rag_can_answer_multihop(corpus)
print(f"  any single corpus chunk links the imager AND the 'Nairobi office': {flat_ok}")
path = shortest_path(graph, probe.start, probe.end)
print(f'  GRAPH traversal ({len(path) - 1} hops): {path_with_relations(graph, path)}')
print(f'  answer entity: {path[-1]}')
assert not flat_ok, 'no single corpus chunk links both ends -> flat RAG cannot connect them'
assert path == list(probe.expected_path), 'traversal path must match the expected chain'
assert path[-1] == 'Nairobi office', 'the traversal must arrive at the responsible office'

query: Where is the team based that leads the satellite carrying the hyperspectral imager?
dense lens: sentence-transformers/all-MiniLM-L6-v2
  flat-RAG top-3 chunks: [1, 0, 10]
    chunk[1]: Helios-7 carries a hyperspectral imager with a ground resolution of 4 meters.
    chunk[0]: The Helios-7 satellite was launched on March 3rd, 2024 from the Kourou spaceport.
    chunk[10]: The Helios-7 ground team spent the afternoon investigating several telemetry errors and console warnings.
  any single corpus chunk links the imager AND the 'Nairobi office': False
  GRAPH traversal (3 hops): hyperspectral imager -[carries]-> Helios-7 -[led_by]-> Dr. Amara Okoye -[based_in]-> Nairobi office
  answer entity: Nairobi office


![Multi-hop: flat RAG retrieves disconnected endpoints (no chunk links both ends → False); the graph
walks the real 3-hop path imager → Helios-7 → Dr. Amara Okoye → Nairobi office. Generated by
`make_figures_09.py`.](../../images/rag09_multihop_vs_flat.png)

![Animated — the multi-hop traversal hopping across the graph edge by edge to the answer entity. The
graph and shortest path are real; the extraction is an illustrative LLM stand-in. Generated by
`make_animation_09.py`.](../../images/rag09_traversal_hop.gif)

## Step 4 — community detection: real modularity-optimizing clusters + the score Q

For global questions we need the graph's thematic clusters. `greedy_modularity_communities` groups
densely-connected entities WITHOUT labels; `modularity` scores the partition. Q>0 means real
structure — and the detected split beats lumping everything into one community.

In [4]:
communities = detect_communities(graph)
q = modularity_score(graph, communities)
print(f'  detected {len(communities)} communities | modularity Q = {q:.4f}')
for i, comm in enumerate(communities):
    print(f'    community {i} ({len(comm)} entities): {sorted(comm)}')
q_blob = modularity_score(graph, [frozenset(graph.nodes())])
print(f'  modularity of the trivial one-blob partition: {q_blob:.4f}')
assert len(communities) == 3, 'this graph should split into 3 communities'
assert abs(q - 0.3200) < 1e-3, 'modularity Q must be 0.3200 (the page/figure value)'
assert q_blob == 0.0, 'the one-blob partition scores Q = 0 exactly'
assert q > q_blob, 'the detected partition must beat the trivial one-blob partition on modularity'

  detected 3 communities | modularity Q = 0.3200
    community 0 (5 entities): ['Ariane 6', 'Helios-7', 'Kourou spaceport', 'hyperspectral imager', 'sun-synchronous orbit']
    community 1 (3 entities): ['Error E-4011', 'ground team', 'telemetry stream']
    community 2 (2 entities): ['Dr. Amara Okoye', 'Nairobi office']
  modularity of the trivial one-blob partition: 0.0000


![Community detection: 3 clusters coloured on the real graph, modularity Q = 0.3200 (detected) vs
0.000 (one blob) — the number that says the split captured real structure. Generated by
`make_figures_09.py`.](../../images/rag09_communities.png)

## Step 5 — Try it: the graph is only as good as the extraction (Pitfall 1, measured)

The multi-hop answer depends on the `led_by` edge (Helios-7 → Dr. Okoye). Suppose the LLM extractor
**missed** that one relation — a single dropped triple. **Predict:** does the imager → office path
break, or reroute? Now run it: delete `led_by` and recompute the shortest path.

In [5]:
# simulate a MISSED extraction: drop the led_by edge (Helios-7 -- Dr. Amara Okoye)
damaged = graph_without_edge(graph, 'Helios-7', 'Dr. Amara Okoye')
intact_path = shortest_path(graph, probe.start, probe.end)
broken_path = shortest_path(damaged, probe.start, probe.end)
print(f'INTACT graph : {len(intact_path) - 1} hops  {path_with_relations(graph, intact_path)}')
print(f'DAMAGED graph: {len(broken_path) - 1} hops  {path_with_relations(damaged, broken_path)}')
okoye_in_intact = 'Dr. Amara Okoye' in intact_path
okoye_in_broken = 'Dr. Amara Okoye' in broken_path
print(f"correct-reasoning entity 'Dr. Amara Okoye' on the path?  intact: {okoye_in_intact}   damaged: {okoye_in_broken}")
# the clean 3-hop answer is destroyed; it detours to 5 hops and NO LONGER passes through Dr. Okoye
assert len(intact_path) - 1 == 3, 'the intact path is 3 hops'
assert len(broken_path) - 1 == 5, 'deleting led_by detours the path to 5 hops (an unrelated shortcut)'
assert okoye_in_intact and not okoye_in_broken, 'the who-leads-it fact is silently lost after the miss'
print('\n-> Pitfall 1 made concrete: ONE missed extraction destroyed the 3-hop answer and rerouted it')
print('   through the WRONG facts (via the fault subgraph) — the who-leads-it reasoning is lost.')

INTACT graph : 3 hops  hyperspectral imager -[carries]-> Helios-7 -[led_by]-> Dr. Amara Okoye -[based_in]-> Nairobi office
DAMAGED graph: 5 hops  hyperspectral imager -[carries]-> Helios-7 -[belongs_to]-> telemetry stream -[appeared_in]-> Error E-4011 -[investigates]-> ground team -[operated_by]-> Nairobi office
correct-reasoning entity 'Dr. Amara Okoye' on the path?  intact: True   damaged: False

-> Pitfall 1 made concrete: ONE missed extraction destroyed the 3-hop answer and rerouted it
   through the WRONG facts (via the fault subgraph) — the who-leads-it reasoning is lost.


## Step 6 — global search: map-reduce over community summaries

The thematic question flat RAG couldn't touch — *"What are the main themes across the corpus?"* — has
a clean answer. Each community gets a summary (an LLM 'community report' in production; a fixed
exemplar here); global search MAPs each community → a partial theme, then REDUCEs them into a final
answer that draws on every community — an aggregate no single chunk could hold.

In [6]:
global_q = 'What are the main themes across the Helios-7 corpus?'
print(f'query: {global_q}')
result = global_search(communities)
print(f'  MAP — one partial theme per community ({len(result.partial_points)} communities):')
for i, point in enumerate(result.partial_points):
    print(f'    community {i}: {point}')
print('  REDUCE — final answer:')
print(f'    {result.final_answer}')
assert len(result.partial_points) == len(communities), 'one partial per community (map covers all)'
assert all(p in result.final_answer for p in result.partial_points), 'reduce must aggregate every community'
print('\n  -> no single chunk holds the themes; the answer is an AGGREGATE over graph communities.')

query: What are the main themes across the Helios-7 corpus?
  MAP — one partial theme per community (3 communities):
    community 0: Spacecraft & mission: Helios-7 itself, its launch from Kourou aboard Ariane 6, the hyperspectral imager it carries, and its sun-synchronous orbit.
    community 1: Operations & faults: telemetry error E-4011, the telemetry stream it appeared in, and the ground team investigating it.
    community 2: People & program leadership: Dr. Amara Okoye, who leads the program, and the Nairobi office she is based in.
  REDUCE — final answer:
    Main themes across the corpus: Spacecraft & mission: Helios-7 itself, its launch from Kourou aboard Ariane 6, the hyperspectral imager it carries, and its sun-synchronous orbit. | Operations & faults: telemetry error E-4011, the telemetry stream it appeared in, and the ground team investigating it. | People & program leadership: Dr. Amara Okoye, who leads the program, and the Nairobi office she is based in.

  -> no single 

![Global search: map-reduce over community summaries — each of the 3 real communities → a partial
theme (map) → aggregated final answer (reduce). Communities are real; summaries are an illustrative
LLM stand-in. Generated by `make_figures_09.py`.](../../images/rag09_global_mapreduce.png)

![Local vs global search: LOCAL (link entities + traverse, multi-hop) vs GLOBAL (detect communities →
summarize → map-reduce, thematic). Generated by `make_figures_09.py`.](../../images/rag09_local_vs_global.png)

## Step 7 — the library one-liners (extraction + summaries need a generative LLM)

In production you don't hand-roll extraction or community reports. Both need an LLM, so they won't
run in this encoder-only env — shown for reference:

```python
# Microsoft GraphRAG — the full local/global pipeline (LLM for extraction + community reports)
#   graphrag index --root ./ragproject
#   graphrag query --method local  --query "..."   # multi-hop / specific-entity
#   graphrag query --method global --query "..."   # whole-corpus / thematic (map-reduce)

# LlamaIndex — PropertyGraphIndex: LLM extracts triples, builds a queryable graph
from llama_index.core import PropertyGraphIndex
index = PropertyGraphIndex.from_documents(documents)   # default: SimpleLLMPathExtractor + ImplicitPathExtractor
retriever = index.as_retriever(include_text=True, similarity_top_k=2)
nodes = retriever.retrieve('Which office runs the satellite carrying the imager?')
```

---

### Recap

- Flat RAG can't answer **multi-hop** (no chunk holds the chain) or **global** (answer is an aggregate).
- **GraphRAG** builds an entity–relation **knowledge graph** (10 entities, 10 relations here).
- **Local search** = traverse (real 3-hop path imager → Helios-7 → Dr. Okoye → Nairobi office).
- **Global search** = detect communities (3 clusters, real Q = 0.320) → summarize → map-reduce.
- It costs an LLM call per chunk to build — a specialist tool; skip it for local factoids.
- Full write-up: [the concept page](../09-GraphRAG.md).